# Automating Anomaly Detection and Monitoring

* This notebook creates a job for running monitoring on the weekly data. The job executes the job_run folder, using job_run/monitoring.py as the entry point.
* To complete the automation workflow, the OCI Data Science Schedule feature should also be used to configure recurring weekly execution.

Conda Environment: generalml_p311_cpu_x86_64_v1

Author: Assaf Rabinowicz, Data Scientist EMEA

In [3]:
from ads.jobs import Job, DataScienceJob, PythonRuntime
from ads import set_auth

set_auth("resource_principal")

In [ ]:
job = (
    Job(name="Monitoring Walmart Anaomaly Detection")
    .with_infrastructure(
        DataScienceJob()
        .with_log_group_id('<your_log_group_ocid>')
        .with_log_id('<your_log_ocid>')
        .with_shape_name("VM.Standard.E4.Flex")
        .with_shape_config_details(memory_in_gbs=4, ocpus=1)
    )
    .with_runtime(
        PythonRuntime()
        .with_service_conda("generalml_p311_cpu_x86_64_v1")
        .with_source("<your_folder_path>") # example: /home/datascience/anomaly_detection_project/regression/src/production_monitoring_automation/job_folder
        .with_working_dir("<folder_name>") # example: 'job_folder' 
        .with_entrypoint("<your_entrypoint_file_name>") # no need full path. Only file name, example: monitoring.py
        .with_output("output", "oci://<bucket_name>@<name_space>") # the output will be deleted after deprovisioning
        .with_environment_variable(INPUT_FILE_NAME="PLACEHOLDER") # this parameters is later customized in the job run using env_var
    )
)
job.create()

In [ ]:
job_run = job.run(
    name="walmart sales monitoring, '2012-08-03.csv'",
    env_var={'INPUT_FILE_NAME': '2012-08-03.csv'} # in every run we can change these values.
)
job_run.watch()